# 🛒 VisualShelf — lanzar la app web desde Colab
Ejecuta las celdas en orden. La última muestra un enlace público `https://xxxx.trycloudflare.com` para compartir: funciona mientras esta sesión de Colab siga abierta.

**Requisito:** la carpeta `app` (con `app.py`, `motor.py`, `static/`, `best.pt` y `parametros_v4.json`) en `MyDrive/proyecto_gondolas/app/`.

In [ ]:
# 1) Montar Drive y revisar los archivos
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
APP = Path('/content/drive/MyDrive/proyecto_gondolas/app')
for p in ('app.py', 'motor.py', 'static/index.html', 'static/app.js', 'static/styles.css', 'best.pt', 'parametros_v4.json'):
    print(('✅' if (APP / p).exists() else '❌ FALTA'), APP / p)

In [ ]:
# 2) Instalar librerías (1-2 min)
!pip install -q ultralytics

In [ ]:
# 3) Iniciar el servidor (queda corriendo en segundo plano)
import subprocess, sys, time
LOG = Path('/content/servidor.log')
try: srv.terminate()
except NameError: pass
srv = subprocess.Popen([sys.executable, 'app.py', '--puerto', '7860'], cwd=str(APP), stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    time.sleep(1)
    if 'VisualShelf listo' in LOG.read_text() or srv.poll() is not None: break
print(LOG.read_text()[-2000:])

In [ ]:
# 4) Crear el enlace público (Cloudflare, sin cuenta)
import re
!wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
TLOG = Path('/content/tunel.log')
try: tunel.terminate()
except NameError: pass
tunel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'], stdout=open(TLOG, 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', TLOG.read_text())
    if m: url = m.group(0); break
if url:
    print('🌐 Enlace público para compartir:', url)
else:
    from google.colab.output import eval_js
    print('No se pudo crear el enlace público. Enlace solo para ti (en este navegador):', eval_js('google.colab.kernel.proxyPort(7860)'))

Para apagar la app: ejecuta `srv.terminate(); tunel.terminate()` o cierra la sesión de Colab. Si algo falla, revisa `/content/servidor.log`.